# Advanced Machine Learning: Modeling & Interpretability
In this notebook, we address the severe class imbalance using SMOTE, evaluate state-of-the-art models like XGBoost, LightGBM, and CatBoost, tune hyperparameters, and use SHAP for model interpretability.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve

# Imblearn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# XAI
import shap
shap.initjs()

## 1. Load Cleaned Data

In [ ]:
df = pd.read_csv('../data/processed/stroke-data-cleaned.csv')
df.head()

## 2. Feature Engineering Setup

In [ ]:
X = df.drop('stroke', axis=1)
y = df['stroke']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

In [ ]:
# Define column types
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
numerical_cols = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease']

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

## 3. Modeling Pipeline with SMOTE

In [ ]:
def evaluate_model(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    print(f"\n{'='*40}\n{name} Evaluation\n{'='*40}")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
### LightGBM Pipeline
lgbm_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LGBMClassifier(random_state=42, verbose=-1))
])
lgbm_pipeline.fit(X_train, y_train)
evaluate_model(lgbm_pipeline, 'LightGBM with SMOTE', X_test, y_test)

In [ ]:
### CatBoost Pipeline
cat_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', CatBoostClassifier(verbose=0, random_state=42))
])
cat_pipeline.fit(X_train, y_train)
evaluate_model(cat_pipeline, 'CatBoost with SMOTE', X_test, y_test)

In [ ]:
### XGBoost Pipeline with Hyperparameter Tuning
# To save time, we do a limited grid search
xgb_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
])

param_grid = {
    'classifier__max_depth': [3, 5],
    'classifier__n_estimators': [100, 200],
    'classifier__learning_rate': [0.05, 0.1]
}

grid_search = GridSearchCV(xgb_pipeline, param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_xgb = grid_search.best_estimator_
print(f"Best XGBoost Parameters: {grid_search.best_params_}")
evaluate_model(best_xgb, 'Tuned XGBoost with SMOTE', X_test, y_test)

## 4. Advanced Evaluation Visualizations

In [ ]:
best_model = best_xgb # Using Tuned XGBoost as our best model

# ROC Curve
y_prob_best = best_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_best)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_prob_best):.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

## 5. Model Interpretability with SHAP

In [ ]:
# Extract preprocessor and feature names
preprocessor_fit = best_model.named_steps['preprocessor']
cat_encoder = preprocessor_fit.named_transformers_['cat']
cat_features_encoded = cat_encoder.get_feature_names_out(categorical_cols)
feature_names = numerical_cols + list(cat_features_encoded)

# Transform X_test to get data for SHAP
X_test_transformed = preprocessor_fit.transform(X_test)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

# Initialize SHAP explainer on the XGBoost classifier step
xgb_classifier = best_model.named_steps['classifier']
explainer = shap.TreeExplainer(xgb_classifier)
shap_values = explainer.shap_values(X_test_df)

# SHAP Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title('SHAP Summary Plot - Feature Impact on Stroke Prediction')
plt.show()

## 6. Save the Final Advanced Model

In [ ]:
model_path = '../models/stroke_prediction_pipeline.pkl'
joblib.dump(best_model, model_path)
print(f"Advanced model pipeline saved to {model_path}")